In [88]:
import pandas as pd
import requests
import hashlib


In [89]:
df=pd.read_csv(r"C:\Users\mikha\Customers\IngeniusData\CustomerDatabase - Carlos.csv")

In [90]:
df["Zip Code"] = (
    df["Zip Code"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

df["City"] = df["City"].astype(str).str.strip().str.upper()
df["State"] = df["State"].astype(str).str.strip().str.upper()
df["Property Address"] = df["Property Address"].astype(str).str.strip()

In [91]:

ADDRESS_URL = "https://geocoding.geo.census.gov/geocoder/locations/address"

def census_geocode(row):
    params = {
        "street": row["Property Address"],
        "city": row["City"],
        "state": row["State"],
        "zip": row["Zip Code"],
        "benchmark": "Public_AR_Current",
        "format": "json"
    }

    try:
        r = requests.get(ADDRESS_URL, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()

        matches = data.get("result", {}).get("addressMatches", [])
        if matches:
            coords = matches[0]["coordinates"]
            return pd.Series([coords["y"], coords["x"]])
    except Exception:
        pass

    return pd.Series([None, None])

df[["Latitude", "Longitude"]] = df.apply(census_geocode, axis=1)

In [92]:
branch_lenders = [
    "NewRez LLC / Caliber",
    "United Wholesale Mortgage",
    "Homepoint Financial",
    "Finance of America Mortgage",
    "Homebridge Financial",
    "Pennymac Loan Services",
    "Princeton Mortgage",
    "Plaza Home Mortgage",
    "Recovco / Sprout",
    "Logan Finance Corp.",
    "American Financial Network"
]

branch_base_df = pd.DataFrame([
    ("WESTFIELD","NJ",40.6589,-74.3474),
    ("LINDEN","NJ",40.6220,-74.2446),
    ("EAST ORANGE","NJ",40.7673,-74.2049),
    ("ELIZABETH","NJ",40.6639,-74.2107),
    ("PLAINFIELD","NJ",40.6337,-74.4074),
    ("TITUSVILLE","NJ",40.3248,-74.8577),
    ("MILFORD","NJ",40.5687,-75.0941),
    ("WEST ORANGE","NJ",40.7987,-74.2390),
    ("BLAIRSTOWN","NJ",40.9826,-74.9616),
    ("EAST NEWARK","NJ",40.7482,-74.1613),
    ("PATERSON","NJ",40.9168,-74.1718),
    ("LEDGEWOOD","NJ",40.8762,-74.6571),
    ("NEW BRUNSWICK","NJ",40.4862,-74.4518),
    ("DUNELLEN","NJ",40.5893,-74.4710),
    ("NEWARK","NJ",40.7357,-74.1724),
    ("RAHWAY","NJ",40.6082,-74.2776),
    ("HEWITT","NJ",41.1762,-74.3568),
    ("LAKE HIAWATHA","NJ",40.8829,-74.3835)
], columns=["City","State","Base_Lat","Base_Lon"])

def branch_jitter(lat, lon, key, scale=0.003):
    h = int(hashlib.md5(key.encode()).hexdigest(), 16)
    lat_offset = ((h % 1000) / 1000 - 0.5) * scale
    lon_offset = (((h // 1000) % 1000) / 1000 - 0.5) * scale
    return lat + lat_offset, lon + lon_offset

branch_rows = []

for _, r in branch_base_df.iterrows():
    city = r["City"]
    state = r["State"]
    h = int(hashlib.md5(city.encode()).hexdigest(), 16)
    branch_count = 1 + (h % 3)

    for i in range(branch_count):
        lender = branch_lenders[(h + i) % len(branch_lenders)]
        key = f"{city}_{lender}"

        lat, lon = branch_jitter(r["Base_Lat"], r["Base_Lon"], key)

        branch_rows.append({
            "City": city,
            "State": state,
            "Branch_Name": f"{lender} – {city.title()} Branch",
            "Branch_Latitude": lat,
            "Branch_Longitude": lon,
            "Branch_Source": "Synthetic"
        })

branch_df = pd.DataFrame(branch_rows)

# =========================
# Explicit branches
# =========================
explicit_branches = pd.DataFrame([
    ("Logan Finance Corp.","PITTSTON","PA",41.3259,-75.7894),
    ("Nexera Holdings","EASTON","PA",40.6884,-75.2207),
    ("NQM Funding","BALTIMORE","MD",39.2904,-76.6122),
    ("Acra Lending / Citadel","MIDDLETOWN","NJ",40.3968,-74.0924),
    ("Acra Lending / Citadel","SCHENECTADY","NY",42.8142,-73.9396)
], columns=["Lender","City","State","Base_Lat","Base_Lon"])

explicit_rows = []

for _, r in explicit_branches.iterrows():
    key = f"{r['Lender']}_{r['City']}_{r['State']}"
    lat, lon = branch_jitter(r["Base_Lat"], r["Base_Lon"], key, scale=0.0025)

    explicit_rows.append({
        "City": r["City"].upper(),
        "State": r["State"],
        "Branch_Name": f"{r['Lender']} – {r['City'].title()} Branch",
        "Branch_Latitude": lat,
        "Branch_Longitude": lon,
        "Branch_Source": "Explicit"
    })

explicit_branch_df = pd.DataFrame(explicit_rows)

In [93]:
# =========================
# Combine branches
# =========================
branch_df = pd.concat(
    [branch_df, explicit_branch_df],
    ignore_index=True
).drop_duplicates(
    subset=["City","Branch_Name"]
)

# =========================
# Join branches
# =========================
df = df.merge(
    branch_df,
    on=["City","State"],
    how="left"
)

In [94]:
worship_df = pd.DataFrame([
    ("PITTSTON","PA",41.3259,-75.7894),
    ("EASTON","PA",40.6884,-75.2207),
    ("HAMBURG","PA",40.5556,-75.9827),
    ("BLOOMSBURY","PA",40.6518,-75.0707),
    ("BALTIMORE","MD",39.2904,-76.6122),
    ("COLUMBIA","MD",39.2037,-76.8610),
    ("SCHENECTADY","NY",42.8142,-73.9396),
    ("UNION","NY",42.0906,-75.9694),
    ("MIDDLETOWN","NJ",40.3968,-74.0924),
    ("KEARNY","NJ",40.7684,-74.1454),
    ("NEWARK","NJ",40.7357,-74.1724),
    ("HARRISON","NJ",40.7465,-74.1563),
    ("PLAINFIELD","NJ",40.6337,-74.4074),
    ("GREAT MEADOWS","NJ",40.8840,-74.9110),
    ("NEW MILFORD","NJ",40.9351,-74.0190),
    ("MANALAPAN","NJ",40.2859,-74.3393),
    ("WAYNE","NJ",40.9254,-74.2765),
    ("FORDS","NJ",40.5293,-74.3154),
    ("FAIR LAWN","NJ",40.9404,-74.1318),
    ("BAYONNE","NJ",40.6687,-74.1143),
    ("MANVILLE","NJ",40.5401,-74.5879),
    ("WHITEHOUSE STATION","NJ",40.6154,-74.7707),
    ("BELLEVILLE","NJ",40.7937,-74.1503),
    ("JERSEY CITY","NJ",40.7178,-74.0431),
    ("ANDOVER","NJ",40.9856,-74.7432),
    ("AVENEL","NJ",40.5807,-74.2776),
    ("RAHWAY","NJ",40.6082,-74.2776),
    ("SCOTCH PLAINS","NJ",40.6554,-74.3896),
    ("ELIZABETH","NJ",40.6639,-74.2107),
    ("BLOOMFIELD","NJ",40.8068,-74.1854),
    ("WESTFIELD","NJ",40.6589,-74.3474),
    ("LINDEN","NJ",40.6220,-74.2446),
    ("EAST ORANGE","NJ",40.7673,-74.2049),
    ("TITUSVILLE","NJ",40.3248,-74.8577),
    ("MILFORD","NJ",40.5687,-75.0941),
    ("WEST ORANGE","NJ",40.7987,-74.2390),
    ("BLAIRSTOWN","NJ",40.9826,-74.9616),
    ("EAST NEWARK","NJ",40.7482,-74.1613),
    ("PATERSON","NJ",40.9168,-74.1718),
    ("LEDGEWOOD","NJ",40.8762,-74.6571)
], columns=["City","State","Base_Lat","Base_Lon"])

In [95]:
worship_types = ["Church","Synagogue","Mosque","Temple"]

worship_df["Worship_Type"] = [
    worship_types[i % len(worship_types)]
    for i in range(len(worship_df))
]

worship_df["Worship_Name"] = worship_df.apply(
    lambda r: f"{r['City'].title()} Community {r['Worship_Type']}",
    axis=1
)

worship_df["Worship_Description"] = worship_df.apply(
    lambda r: f"{r['Worship_Name']} ({r['City'].title()}, {r['State']})",
    axis=1
)

In [96]:
def jitter_coords(lat, lon, key, scale=0.002):
    h = int(hashlib.md5(key.encode()).hexdigest(), 16)
    lat_offset = ((h % 1000) / 1000 - 0.5) * scale
    lon_offset = (((h // 1000) % 1000) / 1000 - 0.5) * scale
    return lat + lat_offset, lon + lon_offset

worship_df[["Worship_Latitude","Worship_Longitude"]] = worship_df.apply(
    lambda r: pd.Series(
        jitter_coords(
            r["Base_Lat"],
            r["Base_Lon"],
            f"{r['City']}_{r['State']}"
        )
    ),
    axis=1
)

worship_df = worship_df[
    [
        "City",
        "State",
        "Worship_Name",
        "Worship_Type",
        "Worship_Description",
        "Worship_Latitude",
        "Worship_Longitude"
    ]
]

In [97]:
df = df.merge(
    worship_df,
    on=["City","State"],
    how="left"
)

In [98]:
df.to_csv(
    r"C:\Users\mikha\Customers\IngeniusData\CustomerDatabase - Carlos_GPS.csv",
    index=False
)
